# Experiment 009a: SmolLM2-360M — Pipeline Smoke Test (1K steps)

Quick validation that the v2 generator data (formulaic templates, hard negatives,
edge cases) works end-to-end. Not expecting big accuracy gains — just verifying
the pipeline before Llama 3.2-1B access is approved.

**Runtime:** T4 is fine for 360M. 1K steps ≈ 15 minutes.

In [ ]:
!pip install -q transformers datasets accelerate torch tensorboard

In [ ]:
import torch, os, subprocess, math
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone & install
if not os.path.exists('tagzeit-gemma-2b'):
    !git clone https://github.com/mgosal/tagzeit-gemma-2b.git
!cd tagzeit-gemma-2b && npm install --silent
print("✅ Ready")

In [ ]:
# Generate data (smaller set — 20K is enough for 1K steps)
!cd tagzeit-gemma-2b && python tools/extract_hard_negatives.py \
    --output data/hard_negatives/complex_tempqa_noroute.jsonl --count 7000

!cd tagzeit-gemma-2b && node experiments/temporal-tagzeit/src/generator_route.js \
    --count 20000 \
    --output data/train/train_routed_009a.jsonl \
    --eval data/eval/eval_routed_009a.jsonl \
    --hard-negatives data/hard_negatives/complex_tempqa_noroute.jsonl

In [ ]:
# Format for SmolLM2
!cd tagzeit-gemma-2b && python tools/format_for_model.py \
    --model_id HuggingFaceTB/SmolLM2-360M-Instruct \
    --input data/train/train_routed_009a.jsonl \
    --output data/train/train_routed_009a_smollm2.jsonl \
    --eval_input data/eval/eval_routed_009a.jsonl \
    --eval_output data/eval/eval_routed_009a_smollm2.jsonl

TRAIN_FILE = 'tagzeit-gemma-2b/data/train/train_routed_009a_smollm2.jsonl'
EVAL_FILE  = 'tagzeit-gemma-2b/data/eval/eval_routed_009a_smollm2.jsonl'

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="auto")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# Domain tokens
ROUTE_TOKENS = [
    "[ROUTE]", "[/ROUTE]", "[NO_ROUTE]",
    "[ROUTE_TIME_ADD]", "[ROUTE_TIME_SUB]", "[ROUTE_DURATION_BETWEEN]",
    "[ROUTE_CALENDAR_SHIFT]", "[ROUTE_TIMEZONE_CONVERT]",
    "[HEAD_TIME]", "[HEAD_DURATION]", "[HEAD_DATE]",
    "[REF_1]", "[REF_2]", "[REF_3]",
]
ROUTE_TOKENS += [f"[ARG_HOUR_{str(h).zfill(2)}]" for h in range(24)]
ROUTE_TOKENS += [f"[ARG_MIN_{str(m).zfill(2)}]" for m in range(60)]
ROUTE_TOKENS += ["[ARG_MON]", "[ARG_TUE]", "[ARG_WED]", "[ARG_THU]", "[ARG_FRI]", "[ARG_SAT]", "[ARG_SUN]"]

num_added = tokenizer.add_tokens(ROUTE_TOKENS)
model.resize_token_embeddings(len(tokenizer))

embed_dim = model.config.hidden_size
with torch.no_grad():
    new_start = len(tokenizer) - num_added
    for i in range(num_added):
        freq = np.array([1.0 / (10000 ** (2 * j / embed_dim)) for j in range(embed_dim)])
        phase = (i + 1) * freq
        init_vec = np.sin(phase) * 0.02
        model.get_input_embeddings().weight[new_start + i] = torch.tensor(init_vec, dtype=torch.float32)

print(f"✅ {MODEL_ID} loaded, {num_added} tokens added")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={'train': TRAIN_FILE, 'eval': EVAL_FILE})

def tokenize_fn(examples):
    result = tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')
print(f"Train: {len(tokenized['train'])}, Eval: {len(tokenized['eval'])}")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "/content/tagzeit-009a"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=1000,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    optim="adamw_torch",
    fp16=(torch.cuda.is_available()),
    bf16=False,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="no",  # No checkpoints for smoke test
    logging_steps=50,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="tensorboard",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['eval'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    processing_class=tokenizer,
)
print("✅ Trainer ready — 1K steps, no checkpoints")

In [ ]:
train_result = trainer.train()
print(f"\nTrain loss: {train_result.training_loss:.4f}")
print(f"Runtime: {train_result.metrics['train_runtime']:.0f}s")

eval_results = trainer.evaluate()
print(f"Eval loss: {eval_results['eval_loss']:.4f}")
print(f"Eval perplexity: {math.exp(eval_results['eval_loss']):.2f}")

In [ ]:
# Quick inference test
test_prompts = [
    "What time is it 1 minute after 23:59?",
    "What time was it 30 minutes before 00:15?",
    "The meeting starts at 14:30 and lasts 45 minutes. When does it end?",
    "What is 42 + 18?",
    "When was The Secret Garden published?",
]

model.eval()
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    response = response.split('<|im_end|>')[0].strip()
    print(f"Q: {prompt}")
    print(f"A: {response}")
    print()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/logs